In [ ]:
!pip install -r https://raw.githubusercontent.com/AHMerrill/housing-project/main/requirements.txt

# Requirements.txt in case the libraries are not present.

There are three main cells to execute here:

**Transformation and Feature Engineering**

The first step applies all necessary data transformations and feature engineering techniques to our holdout set, consistent with the steps previously performed on our training set.


In [93]:
# Transformations for Holdout Set

import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
from sklearn.preprocessing import OneHotEncoder
import re
import ast
from datetime import datetime, timedelta
import warnings
import geopandas as gpd
from shapely.geometry import Point, shape
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from IPython.display import display
import requests

housing_data = pd.read_csv('https://raw.githubusercontent.com/AHMerrill/housing-project/main/austinhouses_holdout.csv')
#housing_data.columns.tolist()


# Lowercase and basic cleanup
housing_data['description'] = housing_data['description'].str.lower().fillna("")

# Define stopwords
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
custom_ignore_words = {
    "austin", "home", "house", "texas", "kitchen", "bedrooms", "2", "3", "master",
    "living", "family", "bath", "bathrooms", "ft", "floor", "bedroom", "sq", "4", "tx", "mo"
}
stop_words = list(ENGLISH_STOP_WORDS.union(custom_ignore_words)) # Convert frozenset to list

# Tokenization and word counting
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words=stop_words)
X_counts = vectorizer.fit_transform(housing_data['description'])
word_counts = pd.DataFrame({
    'word': vectorizer.get_feature_names_out(),
    'count': X_counts.toarray().sum(axis=0)
})
top_words = word_counts.sort_values(by='count', ascending=False).head(25)
# print(top_words)

# Word lists from Zillow
word_lists = {
    'good': [
        'luxurious', 'captivating', 'impeccable', 'stainless', 'basketball',
        'landscaped', 'granite', 'pergola', 'remodel', 'beautiful',
        'gentle', 'spotless', 'tile', 'upgraded', 'updated', 'greenbelt'
    ],
    'bad': [
        'fixer', 'charming', 'motivated seller', 'cozy', 'tlc',
        'cosmetic', 'investment', 'investor', 'potential', 'bargain',
        'opportunity', 'nice', 'bones', 'sold as is'
    ]
}

def count_keywords(texts, keywords):
    return [sum(bool(re.search(rf"\b{re.escape(word)}", txt, flags=re.IGNORECASE)) for txt in texts) for word in keywords]

good_counts = count_keywords(housing_data['description'], word_lists['good'])
bad_counts = count_keywords(housing_data['description'], word_lists['bad'])

good_df = pd.DataFrame({'word': word_lists['good'], 'count': good_counts}).sort_values(by='count', ascending=False)
bad_df = pd.DataFrame({'word': word_lists['bad'], 'count': bad_counts}).sort_values(by='count', ascending=False)

# print(good_df)
# print(bad_df)

# Trimmed word list
word_lists = {
    'good': ['luxurious', 'stainless', 'basketball', 'landscaped',
             'granite', 'pergola', 'remodel', 'beautiful',
             'tile', 'upgraded', 'updated', 'greenbelt'],
    'bad': ['charming', 'cozy', 'investment', 'investor',
            'potential', 'opportunity', 'nice']
}

# One-hot encoding
search_words = {f"good_{w}": w for w in word_lists['good']}
search_words.update({f"bad_{w}": w for w in word_lists['bad']})

for colname, word in search_words.items():
    pattern = re.compile(re.escape(word), flags=re.IGNORECASE)
    housing_data[colname] = housing_data['description'].apply(lambda x: 1 if pattern.search(x) else 0)

# Cleanup + Feature Engineering

housing_data['latest_saledate'] = pd.to_datetime(housing_data['latest_saledate'], errors='coerce')
ref_date = datetime.today() - timedelta(days=(3 * 365))
housing_data['days_since_sale'] = (ref_date - housing_data['latest_saledate']).dt.days
housing_data['yearsOld'] = 2022 - housing_data['yearBuilt']

housing_data = housing_data.drop(columns=[
    'streetAddress', 'latest_saledate', 'latest_salemonth', 'latest_saleyear',
    'yearBuilt', 'homeType', 'description'
], errors='ignore')

# One-hot encode zipcode
zipcode_dummies = pd.get_dummies(housing_data['zipcode'], prefix='zipcode')
rare_zipcodes = ["zipcode78734", "zipcode78742", "zipcode78652", "zipcode78719", "zipcode78738"]
zipcode_dummies = zipcode_dummies.drop(columns=[z for z in rare_zipcodes if z in zipcode_dummies.columns])
housing_data = housing_data.drop(columns='zipcode')
housing_data = pd.concat([housing_data, zipcode_dummies], axis=1)
# Transformed features
housing_data['log_lotSizeSqFt'] = np.log1p(housing_data['lotSizeSqFt'])
housing_data['log_livingAreaSqFt'] = np.log1p(housing_data['livingAreaSqFt'])
housing_data['log_avgSchoolSize'] = np.log1p(housing_data['avgSchoolSize'])
housing_data['yearsOld_sq'] = housing_data['yearsOld'] ** 2
housing_data['avgSchoolRating_sq'] = housing_data['avgSchoolRating'] ** 2


# Create numeric NLP version
good_cols = [col for col in housing_data.columns if col.startswith("good_")]
bad_cols = [col for col in housing_data.columns if col.startswith("bad_")]
housing_data_numeric_NLP = housing_data.copy()
housing_data_numeric_NLP['word_count_good'] = housing_data[good_cols].sum(axis=1)
housing_data_numeric_NLP['word_count_bad'] = housing_data[bad_cols].sum(axis=1)
housing_data_numeric_NLP = housing_data_numeric_NLP.drop(columns=good_cols + bad_cols)

# Export
housing_data_numeric_NLP.to_csv("housing_data_numeric_NLP.csv", index=False)

# Neighborhood detectino
# API endpoint
base_url = "https://data.austintexas.gov/api/odata/v4/inrm-c3ee"

# initialize
all_data = []
url = base_url

# fetch
while url:
    print(f"Fetching: {url}")
    response = requests.get(url)
    data = response.json()

    # adds the thing that it fetched by pulling out the "value"
    all_data.extend(data['value'])

    # check if there's a link to the next page
    url = data.get('@odata.nextLink', None)

# convert all results to a DataFrame
neighborhoods = pd.DataFrame(all_data)

houses_df = housing_data_numeric_NLP

# create Point geometry from longitude and latitude (note order: lon, lat)
# each row's coords are put into a shapely Point file
# wraps the df in GeoDataFrame
# ensures coordinate system is WGS84
houses_df['geometry'] = [Point(lon, lat) for lon, lat in zip(houses_df['longitude'], houses_df['latitude'])]
houses_gdf = gpd.GeoDataFrame(houses_df, geometry='geometry', crs="EPSG:4326")  # WGS 84

# load and convert neighborhood data (from City of Austin code above)
neigh_df = neighborhoods

# convert each dict into a Shapely geometry
# had to go into that file and figure out the geometry was in "the_geom"
neigh_df['geometry'] = neigh_df['the_geom'].apply(shape)
neigh_gdf = gpd.GeoDataFrame(neigh_df, geometry='geometry', crs="EPSG:4326")

# spatial join to add planning_area_name
# if the home is in the neighborhood, it gets a planning_area_name assigned
joined = gpd.sjoin(houses_gdf, neigh_gdf[['planning_area_name', 'geometry']], how='left', predicate='within')

# one-hot encode planning_area_name
# each neighborhood gets a dummy variable with the prefix hood_
joined_encoded = pd.get_dummies(joined, columns=['planning_area_name'], prefix = 'planning_area_name')

# drop geometry and index_right (from spatial join)
joined_encoded = joined_encoded.drop(columns=['geometry', 'index_right','planning_area_name'], errors='ignore')

#joined_encoded.columns.tolist()

# This will be the test set to predict prices.

joined_encoded.drop(columns=['geometry', 'index_right'], errors='ignore').to_csv("housing_data_with_neighborhoods_encoded.csv", index=False)

# Prediction on whole parent training set, which is already transformed and hosted in below URL with same above transofrmations.

housing_rf = pd.read_csv('https://raw.githubusercontent.com/AHMerrill/housing-project/refs/heads/main/housing_data_with_neighborhoods_encoded.csv') # your transformed file name here
housing_rf.columns.tolist()

# Ensuring the column names are uniform for zipcodes in train and the holdout set.

housing_rf.rename(
    columns={col: col.replace('zipcode', 'zipcode_') for col in housing_rf.columns if col.startswith('zipcode') and not col.startswith('zipcode_')},
    inplace=True
)

housing_rf['logLatestPrice'] = np.log(housing_rf['latestPrice'])

# Define predictors and target for training on the full dataset
predictors = housing_rf.drop(columns=['latestPrice', 'logLatestPrice'])
target = housing_rf['logLatestPrice']



Fetching: https://data.austintexas.gov/api/odata/v4/inrm-c3ee



***Random Forest Prediction & Imputation***

Random Forest model to predict and fill in any missing latest price values for our transformed holdout set.

In [99]:
# Loading transformed training dataset

# log transform latestPrice
housing_rf['logLatestPrice'] = np.log(housing_rf['latestPrice'])

# predictors and target
X_train = housing_rf.drop(columns=['latestPrice', 'logLatestPrice'])
y_train = housing_rf['logLatestPrice']

# Set up CV for hyperparameter tuning
kfold = KFold(n_splits=5, shuffle=True, random_state=123)

param_grid = {
    'n_estimators': [100, 300],
    'max_features': ['sqrt', 0.3],
    'max_depth': [None, 20],
    'min_samples_split': [2, 5]
}

rf_model = RandomForestRegressor(random_state=123, oob_score=True, bootstrap=True)

# CV for grid searching
grid = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=kfold,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    refit=True
)

grid.fit(X_train, y_train)

# Best model
best_rf = grid.best_estimator_

'''
# Cross-validated RMSE (true performance estimate) ---
cv_rmse = np.sqrt(-grid.best_score_)
print(f'Cross-validated RMSE: ${cv_rmse:,.2f}')
'''

# Predictions

# Drop columns from joined_encoded that are not in X_train, to avoid issues with fit
cols_to_drop = ['latestPrice'] + [col for col in joined_encoded.columns if col not in X_train.columns]
joined_encoded_for_prediction = joined_encoded.drop(columns=cols_to_drop, errors='ignore')

joined_preds = np.exp(best_rf.predict(joined_encoded_for_prediction))
joined_encoded['latestPrice'] = joined_preds

***Saving Results***

Export the results by saving the CSV files

In [102]:
joined_encoded.to_csv('Final_predictions.csv', index=False)
joined_encoded.head()


,latitude,longitude,propertyTaxRate,garageSpaces,hasAssociation,hasGarage,hasSpa,hasView,latestPrice,numOfPhotos,...,planning_area_name_WEST AUSTIN NEIGHBORHOOD GROUP,planning_area_name_WEST CONGRESS,planning_area_name_WEST OAK HILL,planning_area_name_WEST UNIVERSITY,planning_area_name_WESTGATE,planning_area_name_WINDSOR HILLS,planning_area_name_WINDSOR PARK,planning_area_name_WINDSOR ROAD,planning_area_name_WOOTEN,planning_area_name_ZILKER
0,30.218447,-97.865906,1.98,0,False,False,False,False,405.404672,30,...,False,False,False,False,False,False,False,False,False,False
1,30.188887,-97.990784,2.01,3,True,True,False,False,531.826447,35,...,False,False,False,False,False,False,False,False,False,False
2,30.281693,-97.688553,1.98,1,False,True,False,False,251.617609,2,...,False,False,False,False,False,False,False,False,False,False
3,30.363327,-97.853149,1.98,3,True,True,False,False,752.484385,43,...,False,False,False,False,False,False,False,False,False,False
4,30.167976,-97.887764,1.98,0,True,False,False,True,548.008528,37,...,False,False,False,False,False,False,False,False,False,False
